# 02 — Analysis (laptop; plan.md Stage 4 E2 and later stages)

Reads the merged Kaggle outputs in `results/`. No GPU. Run with the repo's venv (`D:\cvpr\.venv`).
Sections for Stages 5–8 are added by tasks T4 and T7.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "medprune").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from medprune import analysis as A

RESULTS = ROOT / "results"
PROFILE = "t4x2"
MODEL = "lingshu7b"
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.2f}")

In [ ]:
runs = A.load_runs(RESULTS, PROFILE)
display(runs.groupby(["model", "dataset", "method", "budget_pct"]).size().rename("questions").reset_index())

## Stage 4 — kill test (E1) and first diagnostics (E2)

Writes `results/tables/stage4_*.csv`, `stage4_decisions.json` and `results/figures/stage4_keepmaps.pdf`.

In [ ]:
report = A.stage4_report(RESULTS, model=MODEL, profile=PROFILE)
report["decisions"]

**K1** — FastV minus Random on the same questions (points, 95% image-clustered CI). GO if the `fastv - random_l2` CI on `rin_anat_mc` is entirely below 0.

In [ ]:
report["paired"]

**K2** — background retention ratio (Random ≈ 1; above 1 = prefers background). H1 stays the lead if FastV's CI on CT/MRI is above 1.

In [ ]:
report["retention"]

In [ ]:
display(report["accuracy"])
display(report["accuracy_image_dependent"])
display(report["accuracy_by_modality"])

Query dependence (1 = FastV picks the same tokens whatever the question) and high-norm token selection.

In [ ]:
display(report["query_dependence"])
display(report["high_norm"])

Keep-maps: where kept tokens sit, averaged over images (predicted: bottom-right for Lingshu).

In [ ]:
import matplotlib.pyplot as plt

maps = report["keep_maps"]
groups = sorted({g for m in maps.values() for g in m})
fig, axes = plt.subplots(len(maps), len(groups), figsize=(3 * len(groups), 3 * len(maps)), squeeze=False)
for i, (method, by_group) in enumerate(maps.items()):
    for j, group in enumerate(groups):
        ax = axes[i][j]
        if group in by_group:
            ax.imshow(by_group[group], cmap="magma")
        ax.set_title(f"{method} · {group}", fontsize=9)
        ax.set_axis_off()
plt.tight_layout()

After reading: fill the Outcome column of P1–P5 (plan.md §A6), record K1 and K2 in the Work log (Part G), tell the
mentor.

## Stage 5 onwards
Added by tasks T4 (static maps) and T7 (Tables 1–4, Figures 1–5).